# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
# importing the libraries

import os
from openai import AzureOpenAI, OpenAI
from dotenv import load_dotenv
from App.config import headers, azure_endpoint, api_version
from IPython.display import display, Markdown
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)

api_key = os.getenv('cd_api_key')
gemini_api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('e4'):
    print('OpenAI API key found and looks good so far')
else:
    print('API key not found')

if gemini_api_key and gemini_api_key.startswith('AIz'):
    print('Gemini API key found and looks good so far')
else:
    print('API key not found')

In [ ]:
openai = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=azure_endpoint,
    api_key=api_key
)

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=gemini_url, api_key=gemini_api_key)


In [ ]:
system_message = '''
You are a helpful AI assistant for an Airline called FlightyAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
'''
model='gpt-5.4-mini'

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [{'role': 'system', 'content': system_message}] + history + \
                [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(messages=messages, model=model, extra_headers=headers)
    return response.choices[0].message.content


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [ ]:
# Let's make a useful function

ticket_prices = {'mumbai': 5000, 'delhi': 3000, 'kolkata': 5000, 'bangalore': 4000}

def get_ticket_price(destination_city):
    print(f'Tool called for city {destination_city}')
    ticket_price = ticket_prices.get(destination_city.lower(), 'Unknown ticket price')
    return f'The ticket price for the {destination_city} is {ticket_price}'


In [ ]:
get_ticket_price('Mumbai')

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city if requested by the user",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to"
            },
        },
        "required": ["destination_city"],
        "additionalProperites": "False"    
    }
}


In [ ]:
# and this is included in a list of tools:

tools = [{'type': 'function', 'function': price_function}]


In [ ]:
tools

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]

    if tool_call.function.name == 'get_ticket_price':
        arguments = json.loads(tool_call.function.arguments)
        city  = arguments.get('destination_city')
        ticket_price = get_ticket_price(city)
        response = {
            'role': 'tool',
            'content': ticket_price,
            'tool_call_id': tool_call.id
        }

        return response


In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [{'role': 'system', 'content': system_message}] + history + \
                [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(messages=messages, model=model, extra_headers=headers, tools=tools)

    if (response.choices[0].finish_reason == "tool_calls"):
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(messages=messages, model=model, extra_headers=headers, tools=tools)

    return response.choices[0].message.content


In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [ ]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city  = arguments.get('destination_city')
            ticket_price = get_ticket_price(city)
            responses.append({
                'role': 'tool',
                'content': ticket_price,
                'tool_call_id': tool_call.id
            })

    return responses

In [ ]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [{'role': 'system', 'content': system_message}] + history + \
                [{'role': 'user', 'content': message}]
    
    response = openai.chat.completions.create(messages=messages, model=model, extra_headers=headers, tools=tools)

    if (response.choices[0].finish_reason == "tool_calls"):
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.append(responses)
        response = openai.chat.completions.create(messages=messages, model=model, extra_headers=headers, tools=tools)

    return response.choices[0].message.content

In [ ]:
app = gr.ChatInterface(fn=chat)
app.launch()